# 🚀 Transformer + Features CV (Robuste pour Kaggle)

Pipeline GPU pour fine-tuner un modèle Twitter/XLM-R sur le texte + méta-infos et blender avec LightGBM (features structurées).\
- CV stratifié avec pseudo-groupes utilisateurs (évite fuite entre tweets d'un même compte)
- Entraînement mixte précision (fp16 si dispo)
- Ensemble simple (Transformer + LightGBM) avec recherche de poids/threshold sur OOF

In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
import lightgbm as lgb

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch CUDA available: True
GPU name: NVIDIA RTX 4000 Ada Generation


In [2]:
# Caches/paths pour limiter l'écriture disque (utilise /tmp)
import os
from pathlib import Path
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf/datasets"
os.environ["HF_DATASETS_DOWNLOADED_DATASETS"] = "/tmp/hf/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path(os.environ["TRANSFORMERS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)


In [3]:
# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [4]:
# Config centrale
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base"  # Très performant sur tweets multilingues
MAX_LEN = 160
N_SPLITS = 4
EPOCHS = 3
BATCH_SIZE = 16
GRAD_ACC = 2  # effective batch = 32
LR = 2e-5
WEIGHT_DECAY = 0.01
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # cas notebooks/
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submission"
MODEL_DIR = PROJECT_ROOT / "models/transformer_cv"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working dir: {PROJECT_ROOT}")

Working dir: /users/eleves-a/2023/malo.tamalet/influencer-or-observer-1


In [5]:
# Utilitaires texte/méta

def parse_source(source_html: str) -> str:
    if not isinstance(source_html, str):
        return "unknown"
    src = source_html.lower()
    if "iphone" in src:
        return "iphone"
    if "android" in src:
        return "android"
    if "tweetdeck" in src:
        return "tweetdeck"
    if "web" in src or "browser" in src:
        return "web"
    if any(x in src for x in ["buffer", "hootsuite", "socialflow", "sprout", "dlvr.it"]):
        return "bot"
    return "other"

def extract_text(row: pd.Series) -> str:
    if isinstance(row.get("extended_tweet"), dict):
        full = row["extended_tweet"].get("full_text")
        if full:
            return str(full)
    if isinstance(row.get("full_text"), str) and row["full_text"]:
        return str(row["full_text"])
    if isinstance(row.get("text"), str):
        return str(row["text"])
    return ""

def build_input_text(row: pd.Series) -> str:
    user = row.get("user") or {}
    text = extract_text(row)
    desc = (user.get("description") or "")[:160]
    location = (user.get("location") or "")[:80]
    source = parse_source(row.get("source", ""))
    statuses = user.get("statuses_count") or 0
    favourites = user.get("favourites_count") or 0
    listed = user.get("listed_count") or 0
    has_url = 1 if user.get("url") else 0
    is_reply = 1 if row.get("in_reply_to_status_id") is not None else 0
    meta = f"device={source} statuses={statuses} favs={favourites} listed={listed} reply={is_reply} url={has_url} loc={location}"
    return f"{text}[DESC] {desc}[META] {meta}"

def pseudo_user_id(row: pd.Series) -> str:
    user = row.get("user") or {}
    key = "|".join([
        str(user.get("description", ""))[:64],
        str(user.get("profile_image_url_https", "")),
        str(user.get("profile_banner_url", "")),
        str(user.get("statuses_count", 0)),
    ])
    return str(abs(hash(key)) % (10 ** 12))

In [6]:
# Chargement données brutes
train_raw = pd.read_json(DATA_DIR / "train.jsonl", lines=True)
test_raw = pd.read_json(DATA_DIR / "kaggle_test.jsonl", lines=True)

train_raw["text_input"] = train_raw.apply(build_input_text, axis=1)
test_raw["text_input"] = test_raw.apply(build_input_text, axis=1)
train_raw["group"] = train_raw.apply(pseudo_user_id, axis=1)
test_raw["group"] = test_raw.apply(pseudo_user_id, axis=1)

labels = train_raw["label"].astype(int).to_numpy()
groups = train_raw["group"].to_numpy()
test_ids = test_raw["challenge_id"].astype(int).to_numpy()

print(train_raw[["label"]].value_counts(normalize=True))
print(f"Train shape: {train_raw.shape}, Test shape: {test_raw.shape}")

label
0        0.533677
1        0.466323
Name: proportion, dtype: float64
Train shape: (154914, 39), Test shape: (103380, 37)


In [7]:
# HuggingFace Dataset + tokenisation

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_raw[["text_input", "label", "group"]])
test_ds = Dataset.from_pandas(test_raw[["challenge_id", "text_input", "group"]])

# Remove only columns that exist to avoid ValueError if __index_level_0__ is absent
train_remove_cols = [c for c in ["text_input", "group", "__index_level_0__"] if c in train_ds.column_names]
test_remove_cols = [c for c in ["text_input", "group", "challenge_id", "__index_level_0__"] if c in test_ds.column_names]

def tokenize_fn(batch):
    return tokenizer(batch["text_input"], truncation=True, max_length=MAX_LEN)

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_remove_cols)
test_tokenized = test_ds.map(tokenize_fn, batched=True, remove_columns=test_remove_cols)

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


Map: 100%|██████████| 103380/103380 [00:17<00:00, 5913.81 examples/s]


In [8]:
# Fine-tuning CV Transformer

def compute_metrics(eval_pred):
    logits, labels_arr = eval_pred
    preds = logits.argmax(-1)
    acc = accuracy_score(labels_arr, preds)
    return {"accuracy": acc}

skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_probs = np.zeros(len(train_tokenized))
test_probs = np.zeros((len(test_tokenized), 2))

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels, groups)):
    print(f"===== Fold {fold+1}/{N_SPLITS} =====")
    train_split = train_tokenized.select(train_idx.tolist())
    val_split = train_tokenized.select(val_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    out_dir = Path("/tmp/transformer_cv") / f"fold{fold}"
    out_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        gradient_accumulation_steps=GRAD_ACC,
        warmup_ratio=0.1,
        logging_strategy="no",
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=4,
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_split,
        eval_dataset=val_split,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    val_logits = trainer.predict(val_split).predictions
    val_prob = softmax(val_logits, axis=1)[:, 1]
    oof_probs[val_idx] = val_prob

    test_logits = trainer.predict(test_tokenized).predictions
    test_probs += softmax(test_logits, axis=1)

# Moyenne des folds
test_probs /= N_SPLITS

# Optimisation simple du threshold sur OOF
thresholds = np.linspace(0.35, 0.65, 21)
best_thr = 0.5
best_acc = 0
for thr in thresholds:
    acc = accuracy_score(labels, (oof_probs >= thr).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thr = thr

print(f"Transformer OOF accuracy: {best_acc:.4f} @ thr={best_thr:.3f}")
transformer_test_pred = (test_probs[:, 1] >= best_thr).astype(int)


===== Fold 1/4 =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_93304/3302442767.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


===== Fold 2/4 =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_93304/3302442767.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


===== Fold 3/4 =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_93304/3302442767.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


KeyboardInterrupt: 

In [9]:
# Reprendre à partir du fold 3 (index 2) en conservant oof_probs/test_probs existants
splits = list(skf.split(np.zeros(len(labels)), labels, groups))
start = 2  # fold 0 et 1 déjà faits

for fold, (train_idx, val_idx) in enumerate(splits[start:], start=start):
    print(f"===== Fold {fold+1}/{N_SPLITS} (resume) =====")
    train_split = train_tokenized.select(train_idx.tolist())
    val_split = train_tokenized.select(val_idx.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    out_dir = Path("/tmp/transformer_cv") / f"fold{fold}"
    out_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        gradient_accumulation_steps=GRAD_ACC,
        warmup_ratio=0.1,
        logging_strategy="no",
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=4,
        disable_tqdm=False,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_split,
        eval_dataset=val_split,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    val_logits = trainer.predict(val_split).predictions
    oof_probs[val_idx] = softmax(val_logits, axis=1)[:, 1]
    test_logits = trainer.predict(test_tokenized).predictions
    test_probs += softmax(test_logits, axis=1)

# finir par test_probs /= N_SPLITS et calcul du threshold comme avant

===== Fold 3/4 (resume) =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_93304/543553873.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


===== Fold 4/4 (resume) =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_93304/543553873.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


In [12]:
# Moyenne des folds
test_probs /= N_SPLITS

# Optimisation simple du threshold sur OOF
thresholds = np.linspace(0.35, 0.65, 21)
best_thr = 0.5
best_acc = 0
for thr in thresholds:
    acc = accuracy_score(labels, (oof_probs >= thr).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_thr = thr

print(f"Transformer OOF accuracy: {best_acc:.4f} @ thr={best_thr:.3f}")
transformer_test_pred = (test_probs[:, 1] >= best_thr).astype(int)

from xgboost import XGBClassifier

X_feat = np.load(DATA_DIR / "features/X_train_features.npy")
X_test_feat = np.load(DATA_DIR / "features/X_kaggle_features.npy")

xgb_oof = np.zeros(len(labels))
xgb_test = np.zeros(len(X_test_feat))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_feat, labels, groups)):
    print(f"XGB fold {fold+1}/{N_SPLITS}")
    clf = XGBClassifier(
        n_estimators=1200, learning_rate=0.03, max_depth=8,
        min_child_weight=2, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=0.2, gamma=0.1,
        tree_method="hist", device="cuda", predictor="gpu_predictor",
        eval_metric="logloss", random_state=SEED, max_bin=512,
    )
    clf.fit(X_feat[tr_idx], labels[tr_idx],
            eval_set=[(X_feat[val_idx], labels[val_idx])],
            verbose=False)
    xgb_oof[val_idx] = clf.predict_proba(X_feat[val_idx])[:, 1]
    xgb_test += clf.predict_proba(X_test_feat)[:, 1]

xgb_test /= N_SPLITS
best_thr_xgb = max((accuracy_score(labels, (xgb_oof >= thr).astype(int)), thr) for thr in thresholds)[1]


Transformer OOF accuracy: 0.8927 @ thr=0.650
XGB fold 1/4


/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:21:15] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


XGB fold 2/4


/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:21:22] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


XGB fold 3/4


/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:21:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


XGB fold 4/4


/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [21:21:35] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


In [13]:
best_w, best_thr_blend, best_acc_blend = 0.6, best_thr, best_acc
for w in np.linspace(0,1,21):
    blended = w * oof_probs + (1 - w) * xgb_oof
    for thr in thresholds:
        acc = accuracy_score(labels, (blended >= thr).astype(int))
        if acc > best_acc_blend:
            best_acc_blend, best_w, best_thr_blend = acc, w, thr

final_proba = best_w * test_probs[:,1] + (1 - best_w) * xgb_test
final_pred = (final_proba >= best_thr_blend).astype(int)

pd.DataFrame({"ID": test_ids, "Prediction": final_pred}).to_csv(
    SUBMISSION_DIR / "submission_transformer_blend.csv", index=False)


In [16]:
# Méta-blend avec LogisticRegression sur les OOF
import numpy as np
from sklearn.linear_model import LogisticRegression

# si pas encore défini
thresholds = np.linspace(0.35, 0.65, 21)

# Empile Transformer (oof_probs) + XGB (xgb_oof)
stack_X = np.vstack([oof_probs, xgb_oof]).T
stack_test = np.vstack([test_probs[:, 1], xgb_test]).T

meta = LogisticRegression(C=1.0, max_iter=1000)
meta.fit(stack_X, labels)
meta_oof = meta.predict_proba(stack_X)[:, 1]
meta_test = meta.predict_proba(stack_test)[:, 1]

best_thr = 0.5
best_acc = 0
for thr in thresholds:
    acc = accuracy_score(labels, (meta_oof >= thr).astype(int))
    if acc > best_acc:
        best_acc, best_thr = acc, thr
print(f"Meta-logreg OOF acc: {best_acc:.4f} @ thr={best_thr:.3f}")

final_pred = (meta_test >= best_thr).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": final_pred}).to_csv(
    SUBMISSION_DIR / "submission_meta_lr.csv", index=False
)


Meta-logreg OOF acc: 0.9039 @ thr=0.575


In [17]:
# Générer des soumissions avec différents seuils pour le Transformer seul
for thr in [0.55, 0.60, 0.65]:
    pred = (test_probs[:, 1] >= thr).astype(int)
    pd.DataFrame({"ID": test_ids, "Prediction": pred}).to_csv(
        SUBMISSION_DIR / f"submission_transformer_thr_{thr:.2f}.csv", index=False)


In [18]:
# Sauvegarde des submissions
sub_transformer = pd.DataFrame({
    "ID": test_ids,
    "Prediction": transformer_test_pred.astype(int),
})
sub_transformer.to_csv(SUBMISSION_DIR / "submission_transformer_only.csv", index=False)

sub_blend = pd.DataFrame({
    "ID": test_ids,
    "Prediction": final_pred.astype(int),
})
sub_blend.to_csv(SUBMISSION_DIR / "submission_transformer_blend.csv", index=False)

print("Saved:")
print(f" - {SUBMISSION_DIR / 'submission_transformer_only.csv'}")
print(f" - {SUBMISSION_DIR / 'submission_transformer_blend.csv'}")

Saved:
 - /users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/submission/submission_transformer_only.csv
 - /users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/submission/submission_transformer_blend.csv
